# kNN and weighted kNN

**Owner:** Teammate 2

kNN measures distance, so inputs are scaled (`kind="scaled"`). We run it twice: every neighbor counts the same, then closer neighbors count more.

Rules that apply to every notebook: use `prep` and `evaluate` for the split, the preparation and the scoring. Do not write your own. The test set is only touched inside `evaluate.run_feature_steps`.

In [ ]:
# Course      : CS 513 - Data Mining & Machine Learning
# First Name  : <fill in>
# Last Name   : <fill in>
# Id          : <fill in>
# Purpose     : kNN and distance-weighted kNN on the three feature steps.
#
# AI NOTICE   : The starter code in this notebook was generated by AI (Claude).
#               Prompt and context: docs/AI_PROMPT_LOG.md, entry 1.
#               Mark any cell you add with AI help the same way.

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import config
from src import evaluate, plots, prep

df = prep.load_table()
train, test = prep.split_by_building(df)
print(f"{len(df):,} filings, {df['bin'].nunique():,} buildings, "
      f"{df['is_unsafe'].mean():.1%} Unsafe")
print(f"Train: {len(train):,} rows   Test: {len(test):,} rows")

## Train and test on the three feature steps

If this is slow on the real data, tune on a sample: pass `df=df.sample(20000, random_state=config.SEED)` and say so on your slide.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

K_VALUES = [5, 15, 25, 51, 101]

res_u, fit_u = evaluate.run_feature_steps(
    "kNN (uniform)", KNeighborsClassifier(weights="uniform"),
    {"n_neighbors": K_VALUES}, kind="scaled")

res_w, fit_w = evaluate.run_feature_steps(
    "kNN (distance-weighted)", KNeighborsClassifier(weights="distance"),
    {"n_neighbors": K_VALUES}, kind="scaled")

pd.concat([res_u, res_w]).round(3)

## How the score changes with k

In [ ]:
fig, ax = plots.new_axes("Cross-validated score by number of neighbors",
                         xlabel="k (neighbors)", ylabel="CV ROC AUC")
for label, fit, color in (("Uniform", fit_u, plots.SERIES[0]),
                          ("Distance-weighted", fit_w, plots.SERIES[1])):
    cv = pd.DataFrame(fit["3_plus_violations"].cv_results_)
    plots.line(ax, cv["param_model__n_neighbors"].astype(int), cv["mean_test_score"], label, color)
plots.legend(ax)
plt.show()

## To do (Teammate 2)

- Which k won, and did weighting help? One sentence each.
- Why does kNN need scaling? One slide line.
- Note the training size you used if you sampled.